In [1]:
"""
PharmaRAG report figures.
 
Run:  python make_figures.py
Out:  ./figures/fig2_..png  ...  fig6_..png   (300 dpi, light theme, print-safe)
 
Figures 3, 4 and 5 use locked results and run correctly as-is.
Figure 2 needs per-category recall: fill PER_CATEGORY_RECALL below (Appendix B, block 3).
Figure 6 needs the audit log: set AUDIT_LOG_PATH, or it will skip cleanly.
"""
 
import os
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")                      # no display needed; safe on macOS
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

In [2]:
# --------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------
 
OUTDIR = "figures"
AUDIT_LOG_PATH = "monitoring/logs/audit.jsonl"   # <-- adapt for fig 6
 
# Print-safe palette. Colour-blind friendly (Okabe-Ito derived).
C_NEUTRAL = "#4C4C4C"
C_PRIMARY = "#0072B2"
C_ACCENT  = "#D55E00"
C_GREEN   = "#009E73"
C_GREY    = "#BBBBBB"
 
CONFIG_COLORS = {
    "N (no agents)":  "#CC79A7",
    "Baseline":       "#56B4E9",
    "A: expansion":   "#E69F00",
    "B: ctx embed":   "#009E73",
    "C: rerank":      "#0072B2",
}
 
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "-",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})
 
os.makedirs(OUTDIR, exist_ok=True)
 
 
def save(fig, name):
    path = os.path.join(OUTDIR, name)
    fig.savefig(path)
    plt.close(fig)
    print("wrote", path)
 
 

In [3]:
# --------------------------------------------------------------------------
# FIGURE 2 -- Per-category Recall@5 across configurations
# --------------------------------------------------------------------------
# REPLACE with real values from Appendix B block 3.
# Values below are PLACEHOLDERS except dosing under C (0.50), which is recorded.
# Any category left as None is skipped.
 
PER_CATEGORY_RECALL = {
    # category            N      Baseline   A       B       C
    "dosing":            [0.30,  0.42,     0.44,   0.42,   0.50],
    "contraindications": [0.62,  0.74,     0.76,   0.78,   0.88],
    "adverse_reactions": [0.60,  0.72,     0.74,   0.76,   0.86],
    "warnings":          [0.58,  0.70,     0.72,   0.74,   0.84],
    "indications":       [0.66,  0.80,     0.82,   0.84,   0.92],
    "interactions":      [0.55,  0.68,     0.70,   0.72,   0.82],
    "populations":       [0.52,  0.66,     0.68,   0.70,   0.80],
    "patient_style":     [0.57,  0.70,     0.73,   0.75,   0.85],
    "multi_drug":        [0.48,  0.62,     0.64,   0.66,   0.76],
}
PLACEHOLDER_WARNING = True   # set False once real numbers are in
 
 
def figure2():
    cfg_names = list(CONFIG_COLORS.keys())
    cats = sorted(PER_CATEGORY_RECALL, key=lambda c: PER_CATEGORY_RECALL[c][-1])
    y = np.arange(len(cats))
    n = len(cfg_names)
    h = 0.8 / n
 
    fig, ax = plt.subplots(figsize=(9, 0.62 * len(cats) + 2.0))
    for j, cfg in enumerate(cfg_names):
        vals = [PER_CATEGORY_RECALL[c][j] for c in cats]
        offset = (j - (n - 1) / 2) * h
        ax.barh(y + offset, vals, height=h * 0.92,
                color=CONFIG_COLORS[cfg], label=cfg, edgecolor="none")
 
    ax.set_yticks(y)
    ax.set_yticklabels([c.replace("_", " ") for c in cats])
    ax.set_xlim(0, 1.0)
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.set_xlabel("Recall@5")
    ax.set_title("Recall@5 by query category across pipeline configurations")
    ax.grid(axis="y", visible=False)
 
    worst = cats[0]
    ax.annotate(f"weakest category: {worst.replace('_',' ')} "
                f"({PER_CATEGORY_RECALL[worst][-1]:.2f} under C)",
                xy=(PER_CATEGORY_RECALL[worst][-1], 0),
                xytext=(0.62, 0.6), textcoords="axes fraction",
                fontsize=9, color=C_NEUTRAL,
                arrowprops=dict(arrowstyle="->", color=C_NEUTRAL, lw=0.9))
 
    ax.legend(ncol=n, loc="upper center", bbox_to_anchor=(0.5, -0.10 - 0.02 * len(cats) / 5))
 
    if PLACEHOLDER_WARNING:
        fig.text(0.5, 1.005, "PLACEHOLDER DATA -- replace via Appendix B block 3",
                 ha="center", fontsize=8, color=C_ACCENT, weight="bold")
    save(fig, "fig2_recall_by_category.png")

In [4]:

# --------------------------------------------------------------------------
# FIGURE 3 -- Groundedness vs unsafe emission  (the money figure)
# --------------------------------------------------------------------------
 
def figure3():
    configs = ["N\n(no agents)", "Baseline", "C\n(final)"]
    groundedness = [0.945, 0.960, 0.932]
    unsafe = [1.000, 0.185, 0.111]
    x = np.arange(len(configs))
 
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
 
    ax1.bar(x, groundedness, width=0.55, color=C_PRIMARY, edgecolor="none")
    ax1.plot(x, groundedness, color=C_NEUTRAL, lw=1.4, marker="o",
             ms=5, zorder=5, label="groundedness")
    for xi, v in zip(x, groundedness):
        ax1.text(xi, v + 0.035, f"{v:.3f}", ha="center", fontsize=9.5, weight="bold")
    ax1.set_title("Groundedness (answered queries)", pad=12)
    ax1.set_ylabel("Rate")
 
    ax2.bar(x, unsafe, width=0.55, color=C_ACCENT, edgecolor="none")
    ax2.plot(x, unsafe, color=C_NEUTRAL, lw=1.4, marker="o", ms=5, zorder=5)
    for xi, v in zip(x, unsafe):
        ax2.text(xi, v + 0.035, f"{v:.3f}", ha="center", fontsize=9.5, weight="bold")
    ax2.set_title("Unsafe emission rate (27 should-refuse queries)", pad=12)
 
    for ax in (ax1, ax2):
        ax.set_xticks(x)
        ax.set_xticklabels(configs)
        ax.set_ylim(0, 1.10)          # CRITICAL: identical, do not autoscale
        ax.set_yticks(np.arange(0, 1.01, 0.2))
        ax.grid(axis="x", visible=False)
 
    ax1.annotate("statistically flat",
                 xy=(1, 0.960), xytext=(0.30, 0.62), textcoords="axes fraction",
                 fontsize=9, color=C_NEUTRAL,
                 arrowprops=dict(arrowstyle="-", color=C_NEUTRAL, lw=0.9))
    ax2.annotate("architectural floor:\nno refusal mechanism exists",
                 xy=(0, 1.000), xytext=(0.28, 0.70), textcoords="axes fraction",
                 fontsize=9, color=C_NEUTRAL,
                 arrowprops=dict(arrowstyle="->", color=C_NEUTRAL, lw=0.9))
 
    fig.suptitle("Groundedness does not track safety", fontsize=13, y=1.00)
    fig.tight_layout()
    save(fig, "fig3_groundedness_vs_safety.png")

In [5]:

# --------------------------------------------------------------------------
# FIGURE 4 -- Semantic weight sweep (RRF degeneracy)
# --------------------------------------------------------------------------
 
def figure4():
    w = np.array([0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    recall = np.array([0.655, 0.707, 0.707, 0.724, 0.741, 0.776, 0.776])
    dense_only = 0.776
 
    fig, ax = plt.subplots(figsize=(8.4, 4.8))
 
    ax.axhline(dense_only, ls="--", lw=1.2, color=C_NEUTRAL, zorder=1)
    ax.text(0.415, dense_only + 0.006, "dense-only ceiling (0.776)",
            fontsize=9, color=C_NEUTRAL)
 
    ax.axvspan(0.9, 1.0, color=C_GREY, alpha=0.30, zorder=0)
    ax.text(0.95, 0.663, "plateau", ha="center", fontsize=9, color=C_NEUTRAL)
 
    ax.plot(w, recall, color=C_PRIMARY, lw=2.0, marker="o", ms=6, zorder=3)
 
    i = int(np.where(np.isclose(w, 0.6))[0][0])
    ax.scatter([w[i]], [recall[i]], s=150, facecolors="none",
               edgecolors=C_ACCENT, lw=2.0, zorder=4)
    ax.annotate("deployed configuration\n(0.6 / 0.4)",
                xy=(w[i], recall[i]), xytext=(0.60, 0.30),
                textcoords="axes fraction", fontsize=9, color=C_ACCENT,
                arrowprops=dict(arrowstyle="->", color=C_ACCENT, lw=1.1))
 
    ax.set_xlabel("Semantic weight in RRF fusion  ($w_s$; lexical weight = $1 - w_s$)")
    ax.set_ylabel("Recall@5")
    ax.set_xlim(0.38, 1.02)
    ax.set_ylim(0.63, 0.80)
    ax.set_xticks(w)
    ax.set_title("Recall rises monotonically with semantic weight:\n"
                 "the lexical retriever contributes no novel candidates at any setting")
    save(fig, "fig4_semantic_weight_sweep.png")
 
 

In [6]:

# --------------------------------------------------------------------------
# FIGURE 5 -- Latency breakdown
# --------------------------------------------------------------------------
 
def figure5():
    stages = ["Query routing", "Retrieval + rerank", "Generation", "Validation"]
    colors = [C_GREEN, C_PRIMARY, C_ACCENT, "#CC79A7"]
    baseline = [2.15, 0.082, 10.3, 0.55]
    config_c = [2.15, 2.27, 10.3, 0.55]
    rows = ["Baseline\n(no rerank)", "Config C\n(rerank)"]
    data = np.array([baseline, config_c])
 
    fig, ax = plt.subplots(figsize=(9.2, 3.4))
    left = np.zeros(len(rows))
    for k, (stage, col) in enumerate(zip(stages, colors)):
        vals = data[:, k]
        ax.barh(rows, vals, left=left, color=col, label=stage,
                height=0.5, edgecolor="white", linewidth=0.8)
        for r in range(len(rows)):
            if vals[r] >= 0.9:
                ax.text(left[r] + vals[r] / 2, r, f"{vals[r]:.2f}s",
                        ha="center", va="center", fontsize=9,
                        color="white", weight="bold")
        left += vals
 
    for r in range(len(rows)):
        ax.text(left[r] + 0.25, r, f"total {left[r]:.2f}s",
                va="center", fontsize=9.5, weight="bold", color=C_NEUTRAL)
 
    ax.set_xlabel("Latency (seconds)")
    ax.set_xlim(0, left.max() * 1.16)
    ax.grid(axis="y", visible=False)
    ax.set_title("Generation dominates end-to-end latency; "
                 "reranking and validation are cheap by comparison")
    ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.28))
    save(fig, "fig5_latency_breakdown.png")
 
 
# --------------------------------------------------------------------------
# FIGURE 6 -- Confidence distribution by decision tier (needs audit log)
# --------------------------------------------------------------------------
 
def figure6():
    if not os.path.exists(AUDIT_LOG_PATH):
        print(f"skip fig6: no audit log at {AUDIT_LOG_PATH} "
              f"(set AUDIT_LOG_PATH at top of script)")
        return
 
    recs = []
    with open(AUDIT_LOG_PATH) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            conf = r.get("confidence")
            dec = r.get("decision")
            if conf is None or dec is None:
                continue
            recs.append((float(conf), str(dec)))
 
    if not recs:
        print("skip fig6: audit log parsed but no usable "
              "'confidence'/'decision' fields found")
        return
 
    conf = np.array([c for c, _ in recs])
    dec = np.array([d for _, d in recs])
 
    tiers = [("INSUFFICIENT_EVIDENCE", C_ACCENT),
             ("ANSWER_WITH_CAUTION", "#E69F00"),
             ("ANSWER", C_PRIMARY)]
 
    fig, ax = plt.subplots(figsize=(8.6, 4.4))
    bins = np.linspace(0, 1, 31)
    for name, col in tiers:
        sub = conf[dec == name]
        if sub.size == 0:
            continue
        ax.hist(sub, bins=bins, alpha=0.70, color=col,
                label=f"{name} (n={sub.size})", edgecolor="white", linewidth=0.5)
 
    for t in (0.45, 0.65):
        ax.axvline(t, ls="--", lw=1.2, color=C_NEUTRAL)
        ax.text(t, ax.get_ylim()[1] * 0.96, f" τ={t}", fontsize=9, color=C_NEUTRAL)
 
    ax.set_xlabel("Refusal Guard confidence  $C$")
    ax.set_ylabel("Number of queries")
    ax.set_xlim(0, 1)
    ax.set_title("Confidence distribution by release decision")
    ax.legend()
    save(fig, "fig6_confidence_distribution.png")
 
 
if __name__ == "__main__":
    figure2()
    figure3()
    figure4()
    figure5()
    figure6()
    print("\nDone. Figures in ./figures/")

wrote figures/fig2_recall_by_category.png
wrote figures/fig3_groundedness_vs_safety.png
wrote figures/fig4_semantic_weight_sweep.png
wrote figures/fig5_latency_breakdown.png
skip fig6: no audit log at monitoring/logs/audit.jsonl (set AUDIT_LOG_PATH at top of script)

Done. Figures in ./figures/


Appendix B

In [1]:
import chromadb, pandas as pd
from collections import Counter

client = chromadb.PersistentClient(path="./chroma_db")   # adapt
col = client.get_collection("pharmarag_spl")             # adapt
meta = col.get(include=["metadatas"])["metadatas"]

df = pd.DataFrame(meta)
print("Total chunks:", len(df))
print("Distinct drugs:", df["drug_name"].nunique())

per_drug = (df.groupby("drug_name")
              .size().reset_index(name="chunks")
              .sort_values("chunks", ascending=False))
print(per_drug.to_markdown(index=False))

per_section = (df.groupby("section_name")
                 .size().reset_index(name="chunks")
                 .sort_values("chunks", ascending=False))
print(per_section.to_markdown(index=False))

print("\nDrugs with zero chunks in a core section (corpus gaps):")
core = ["boxed_warning","contraindications","warnings_and_precautions",
        "dosage_and_administration","adverse_reactions"]
for d in df["drug_name"].unique():
    have = set(df[df["drug_name"]==d]["section_name"])
    missing = [s for s in core if s not in have]
    if missing:
        print(" ", d, "->", missing)

/Users/omkarbadadale/Documents/MS Data Science/Spring 26/Capstone Project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


InvalidCollectionException: Collection pharmarag_spl does not exist.

In [3]:
"""
PharmaRAG architecture diagram (Figure 1).

Run:  python make_architecture.py
Out:  figures/fig1_architecture.png   (300 dpi, light theme, print-safe)

Pure matplotlib, no graphviz dependency.
"""

import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

OUTDIR = "figures"
os.makedirs(OUTDIR, exist_ok=True)

# palette (matches make_figures.py)
C_INGEST = "#D9D9D9"     # offline ingestion, greyed
C_RETR   = "#9ECAE8"     # retrieval stages
C_GEN    = "#F0C27A"     # generation
C_AGENT  = "#0072B2"     # the three agents, filled dark
C_OUT_OK = "#009E73"
C_OUT_CAU= "#E69F00"
C_OUT_NO = "#D55E00"
C_AUDIT  = "#EDEDED"
C_EDGE   = "#3A3A3A"
C_TEXT   = "#1A1A1A"

plt.rcParams.update({
    "font.size": 9,
    "figure.facecolor": "white",
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

fig, ax = plt.subplots(figsize=(13.5, 8.2))
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.axis("off")


def box(x, y, w, h, label, sub=None, fc=C_RETR, tc=C_TEXT,
        fontsize=9, weight="normal", radius=1.4):
    """Rounded box with centred label and optional smaller sub-label."""
    p = FancyBboxPatch((x, y), w, h,
                       boxstyle=f"round,pad=0,rounding_size={radius}",
                       linewidth=1.1, edgecolor=C_EDGE, facecolor=fc, zorder=3)
    ax.add_patch(p)
    if sub:
        ax.text(x + w / 2, y + h * 0.62, label, ha="center", va="center",
                fontsize=fontsize, color=tc, weight=weight, zorder=4)
        ax.text(x + w / 2, y + h * 0.27, sub, ha="center", va="center",
                fontsize=fontsize - 1.4, color=tc, alpha=0.85, zorder=4)
    else:
        ax.text(x + w / 2, y + h / 2, label, ha="center", va="center",
                fontsize=fontsize, color=tc, weight=weight, zorder=4)
    return (x, y, w, h)


def arrow(p1, p2, style="-|>", ls="-", color=C_EDGE, lw=1.3,
          rad=0.0, zorder=2):
    a = FancyArrowPatch(p1, p2, arrowstyle=style, linestyle=ls,
                        color=color, linewidth=lw, mutation_scale=13,
                        connectionstyle=f"arc3,rad={rad}",
                        shrinkA=1, shrinkB=1, zorder=zorder)
    ax.add_patch(a)


def right(b):   return (b[0] + b[2], b[1] + b[3] / 2)
def left(b):    return (b[0], b[1] + b[3] / 2)
def top(b):     return (b[0] + b[2] / 2, b[1] + b[3])
def bottom(b):  return (b[0] + b[2] / 2, b[1])


def band(y, h, label, color="#FAFAFA", label_x=3.0):
    ax.add_patch(Rectangle((1.5, y), 97, h, facecolor=color,
                           edgecolor="#DDDDDD", linewidth=0.9, zorder=0))
    ax.text(label_x, y + h - 2.4, label, fontsize=8.6, style="italic",
            color="#666666", ha="left", va="center", zorder=1)


# ==========================================================================
# BAND A -- offline ingestion
# ==========================================================================
band(78, 19, "OFFLINE INGESTION (run once per corpus build)")

BH, BY = 8.5, 82
a1 = box(4,    BY, 15.5, BH, "DailyMed SPL", "XML, SetID-pinned", fc=C_INGEST)
a2 = box(23.5, BY, 15.5, BH, "LOINC parse", "section metadata", fc=C_INGEST)
a3 = box(43,   BY, 15.5, BH, "Chunking", "500 tok / 50 overlap", fc=C_INGEST)
a4 = box(62.5, BY, 15.5, BH, "PubMedBERT", "768-d embeddings", fc=C_INGEST)
a5 = box(82,   BY, 14,   BH, "ChromaDB", "723 chunks", fc=C_INGEST)

for s, t in [(a1, a2), (a2, a3), (a3, a4), (a4, a5)]:
    arrow(right(s), left(t))

# ==========================================================================
# BAND B -- online retrieval
# ==========================================================================
band(45, 29, "ONLINE: RETRIEVAL")

q  = box(4,  56, 15.5, 8.5, "User query", fc="white")
r1 = box(23.5, 56, 15.5, 8.5, "Query Router", "query \u2192 SPL section",
         fc=C_AGENT, tc="white", weight="bold")

# parallel retrievers
d1 = box(43, 61.5, 15.5, 7.0, "Dense search", "cosine, ChromaDB", fc=C_RETR)
d2 = box(43, 47.5, 15.5, 7.0, "BM25", "lexical", fc=C_RETR)

fu = box(62.5, 54.5, 15.5, 8.5, "Weighted RRF", "k=60, 0.6 / 0.4", fc=C_RETR)
rr = box(82, 54.5, 14, 8.5, "Cross-encoder", "top-20 \u2192 top-5", fc=C_RETR)

arrow(right(q), left(r1))
arrow(right(r1), left(d1), rad=-0.18)
arrow(right(r1), left(d2), rad=0.18)
arrow(right(d1), left(fu), rad=0.18)
arrow(right(d2), left(fu), rad=-0.18)
arrow(right(fu), left(rr))

# index feeds dense search
arrow(bottom(a5), (89, 76.0), ls=(0, (4, 3)), style="-", color="#8A8A8A")
arrow((89, 76.0), (50.75, 76.0), ls=(0, (4, 3)), style="-", color="#8A8A8A")
arrow((50.75, 76.0), top(d1), ls=(0, (4, 3)), style="-|>", color="#8A8A8A")
ax.text(70, 77.3, "vector index", fontsize=8, color="#8A8A8A", ha="center")

# ==========================================================================
# BAND C -- generation and governance
# ==========================================================================
band(6, 33, "ONLINE: GENERATION AND GOVERNANCE", label_x=20.0)

gen = box(4, 27, 19, 9.0, "Gemma 3 12B", "Ollama, local, T=0", fc=C_GEN)
ev  = box(28, 27, 19, 9.0, "Evidence Validator",
          "per-sentence, \u03c4=0.35", fc=C_AGENT, tc="white", weight="bold")
gd  = box(52, 27, 19, 9.0, "Refusal Guard",
          "C = .25r + .55g + .20n", fc=C_AGENT, tc="white", weight="bold")

arrow(right(gen), left(ev))
arrow(right(ev), left(gd))

# rerank output wraps down into generation
arrow((89, 54.5), (89, 42.0), style="-")
arrow((89, 42.0), (13.5, 42.0), style="-")
arrow((13.5, 42.0), top(gen), style="-|>")
ax.text(51, 43.3, "top-5 chunks as generation context",
        fontsize=8.4, color="#555555", ha="center")

# three outcomes
o1 = box(76, 33.5, 20, 6.4, "ANSWER", "C \u2265 0.65", fc=C_OUT_OK, tc="white")
o2 = box(76, 26.0, 20, 6.4, "ANSWER WITH CAUTION", "0.45 \u2264 C < 0.65",
         fc=C_OUT_CAU, fontsize=8.4)
o3 = box(76, 18.5, 20, 6.4, "INSUFFICIENT EVIDENCE", "C < 0.45",
         fc=C_OUT_NO, tc="white", fontsize=8.4)

arrow(right(gd), left(o1), rad=-0.16)
arrow(right(gd), left(o2))
arrow(right(gd), left(o3), rad=0.16)

# ==========================================================================
# AUDIT LOG
# ==========================================================================
au = box(4, 9.5, 92, 6.6, "Audit log (JSONL)",
         "request id  \u00b7  routed section  \u00b7  chunk ids + scores  \u00b7  "
         "per-stage latency  \u00b7  groundedness  \u00b7  confidence  \u00b7  decision",
         fc=C_AUDIT, fontsize=9)

for src in [bottom(r1), bottom(gen), bottom(ev), bottom(gd)]:
    arrow(src, (src[0], 16.1), ls=(0, (3, 3)),
          color="#9A9A9A", lw=0.95, style="-|>")

# ==========================================================================
# LEGEND
# ==========================================================================
handles = [
    ("Agentic governance layer", C_AGENT),
    ("Retrieval stage", C_RETR),
    ("Generation", C_GEN),
    ("Offline ingestion", C_INGEST),
]
for i, (lab, col) in enumerate(handles):
    x0 = 4 + i * 24
    ax.add_patch(FancyBboxPatch((x0, 2.4), 3.2, 2.6,
                                boxstyle="round,pad=0,rounding_size=0.6",
                                facecolor=col, edgecolor=C_EDGE,
                                linewidth=1.0, zorder=3))
    ax.text(x0 + 4.2, 3.7, lab, fontsize=8.8, va="center", color=C_TEXT)

ax.text(50, 97.5, "PharmaRAG system architecture",
        ha="center", va="center", fontsize=14, weight="bold", color=C_TEXT)

out = os.path.join(OUTDIR, "fig1_architecture.png")
fig.savefig(out)
plt.close(fig)
print("wrote", out)

wrote figures/fig1_architecture.png


In [4]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")   # try "./chroma", "./db", "./data/chroma" if empty
cols = client.list_collections()
print([c.name for c in cols])
for c in cols:
    print(c.name, c.count())
    print("  sample metadata:", c.peek(1)["metadatas"])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


[]


In [5]:
"""
Output A: Corpus composition.
Reads data/processed/chunks.jsonl (723 chunks, real schema: chunk_id, drug_name,
generic_name, set_id, section_name, loinc_code, chunk_index, total_chunks, text).
Run from project root: python report_analysis/A_corpus_composition.py
"""
import json, csv, collections, pathlib

ROOT = pathlib.Path(__file__).parent.parent
OUT = ROOT / "report_analysis" / "output"
OUT.mkdir(parents=True, exist_ok=True)

chunks = [json.loads(l) for l in open(ROOT / "data/processed/chunks.jsonl")]

# A1: totals
total_chunks = len(chunks)
distinct_drugs = len(set(c["drug_name"] for c in chunks))
with open(OUT / "A1_totals.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["metric", "value"])
    w.writerow(["total_chunks", total_chunks])
    w.writerow(["distinct_drugs", distinct_drugs])

# A2: chunks per drug, descending
by_drug = collections.Counter(c["drug_name"] for c in chunks)
with open(OUT / "A2_chunks_per_drug.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["drug_name", "chunk_count"])
    for d, n in by_drug.most_common():
        w.writerow([d, n])

# A3: chunks per SPL section, descending
by_section = collections.Counter(c["section_name"] for c in chunks)
with open(OUT / "A3_chunks_per_section.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["section_name", "chunk_count"])
    for s, n in by_section.most_common():
        w.writerow([s, n])

# A4: core-section gap check (boxed_warning, contraindications, warnings_and_precautions,
# dosage_and_administration, adverse_reactions)
CORE = ["boxed_warning", "contraindications", "warnings_and_precautions",
        "dosage_and_administration", "adverse_reactions"]
by_drug_sections = collections.defaultdict(set)
for c in chunks:
    by_drug_sections[c["drug_name"]].add(c["section_name"])
with open(OUT / "A4_core_section_gaps.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["drug_name", "missing_core_sections"])
    for d in sorted(by_drug_sections):
        missing = [s for s in CORE if s not in by_drug_sections[d]]
        if missing:
            w.writerow([d, ";".join(missing)])

# A5: full drug x section presence/count matrix (28 x 9)
ALL_SECTIONS = sorted(by_section.keys())
by_drug_counts = collections.defaultdict(collections.Counter)
for c in chunks:
    by_drug_counts[c["drug_name"]][c["section_name"]] += 1
with open(OUT / "A5_drug_section_matrix.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["drug_name"] + ALL_SECTIONS)
    for d in sorted(by_drug_counts):
        w.writerow([d] + [by_drug_counts[d].get(s, 0) for s in ALL_SECTIONS])

print(f"total_chunks={total_chunks} distinct_drugs={distinct_drugs}")
print("Wrote A1-A5 to", OUT)

NameError: name '__file__' is not defined